In [ ]:
!pip install pandas openpyxl sentence-transformers chromadb scikit-learn numpy torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
import chromadb

In [ ]:
# Path data di Drive
DATA_PATH = "/content/drive/MyDrive/skripsian/data"
TKPI_PATH = f"{DATA_PATH}/tkpi-2019.xlsx"

In [ ]:
import pandas as pd
# Load TKPI
tkpi_df = pd.read_excel(TKPI_PATH, sheet_name="Komposisi Pangan")

# hapus baris pada dataframe dengan NaN
tkpi_df = tkpi_df.dropna(how='all')

# Set nama kolom DataFrame
tkpi_df.columns = ['KODE','NAMA BAHAN','SUMBER','AIR (g)','ENERGI (Kal)','PROTEIN (g)','LEMAK (g)','KH (g)','SERAT (g)','ABU (g)','KALSIUM (mg)','FOSFOR (mg)','BESI (mg)','NATRIUM (mg)','KALIUM (mg)','TEMBAGA (mg)','SENG (mg)','RETINOL (mcg)','B-KAR (mcg)','KAR-TOTAL (mcg)','THIAMIN (mg)','RIBOFLAVIN (mg)','NIASIN (mg)','VIT-C (mg)','BDD (%)']

In [ ]:
print("TKPI berhasil load dari Drive!")
print("Total baris:", len(tkpi_df))
display(tkpi_df.head())

TKPI berhasil load dari Drive!
Total baris: 1177


,KODE,NAMA BAHAN,SUMBER,AIR (g),ENERGI (Kal),PROTEIN (g),LEMAK (g),KH (g),SERAT (g),ABU (g),...,TEMBAGA (mg),SENG (mg),RETINOL (mcg),B-KAR (mcg),KAR-TOTAL (mcg),THIAMIN (mg),RIBOFLAVIN (mg),NIASIN (mg),VIT-C (mg),BDD (%)
2,TUNGGAL/SINGLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AR001,"Beras giling, mentah",KZGMI-2001,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.10,0.5,0,0,0,0.20,0.08,2.6,0,100.0
4,AR002,"Beras giling var pelita, mentah",KZGPI- 1990,11.4,369.0,9.5,1.4,77.1,0.4,0.6,...,0.00,0.0,0,0,0,0.26,0.00,0.0,0,100.0
5,AR003,"Beras giling var \nrojolele, mentah",KZGPI- 1990,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.14,0.1,NaN,0,80,0.20,0.02,1.5,0,100.0
6,AR004,"Beras hitam, mentah",KZGMI-2001,12.9,351.0,8.0,1.3,76.9,20.1,0.9,...,0.10,1.6,0,0,0,0.21,0.06,0.0,0,100.0


In [ ]:
# Hapus baris yang NAMA BAHAN NaN atau "TUNGGAL/SINGLE"
tkpi_df = tkpi_df.dropna(subset=['NAMA BAHAN'])
tkpi_df = tkpi_df[tkpi_df['NAMA BAHAN'].str.strip() != "TUNGGAL/SINGLE"]

# Reset index
tkpi_df = tkpi_df.reset_index(drop=True)

In [ ]:
import re

def deep_clean_nama_bahan(text):
    if pd.isna(text):
        return ""
    # Ganti \n dan \r dengan spasi
    text = str(text).replace('\n', ' ').replace('\r', ' ')
    # Ganti multiple spasi jadi satu spasi
    text = re.sub(r'\s+', ' ', text)
    # Normalisasi koma (spasi sebelum/sesudah koma)
    text = re.sub(r'\s*,\s*', ', ', text)
    # Strip lagi
    text = text.strip()
    return text

# Terapkan deep cleaning ke kolom nama bahan
tkpi_df['NAMA BAHAN'] = tkpi_df['NAMA BAHAN'].apply(deep_clean_nama_bahan)

# Cek hasilnya (penting! harus kamu lihat)
print("Contoh 20 nama bahan setelah deep cleaning:")
print(tkpi_df['NAMA BAHAN'].head(20).tolist())

display(tkpi_df.head())  # lihat apakah nama sudah rapi

Contoh 20 nama bahan setelah deep cleaning:
['Beras giling, mentah', 'Beras giling var pelita, mentah', 'Beras giling var rojolele, mentah', 'Beras hitam, mentah', 'Beras jagung kuning, kering, mentah', 'Beras jagung putih, kering, mentah', 'Beras ketan hitam tumbuk, mentah', 'Beras ketan putih tumbuk, mentah', 'Beras ladang, mentah', 'Beras menir, mentah', 'Beras parboiled', 'Beras tumbuk, mentah', 'Beras tumbuk merah, mentah', 'Cantel, mentah', 'Jagung muda, kuning, mentah', 'Jagung kuning pipil, kering, mentah', 'Jagung pipil var, harapan, kering', 'Jagung pipil var, metro, kering', 'Jali, mentah', 'Jawawut, mentah']


,KODE,NAMA BAHAN,SUMBER,AIR (g),ENERGI (Kal),PROTEIN (g),LEMAK (g),KH (g),SERAT (g),ABU (g),...,TEMBAGA (mg),SENG (mg),RETINOL (mcg),B-KAR (mcg),KAR-TOTAL (mcg),THIAMIN (mg),RIBOFLAVIN (mg),NIASIN (mg),VIT-C (mg),BDD (%)
0,AR001,"Beras giling, mentah",KZGMI-2001,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.10,0.5,0,0,0,0.20,0.08,2.6,0,100.0
1,AR002,"Beras giling var pelita, mentah",KZGPI- 1990,11.4,369.0,9.5,1.4,77.1,0.4,0.6,...,0.00,0.0,0,0,0,0.26,0.00,0.0,0,100.0
2,AR003,"Beras giling var rojolele, mentah",KZGPI- 1990,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.14,0.1,NaN,0,80,0.20,0.02,1.5,0,100.0
3,AR004,"Beras hitam, mentah",KZGMI-2001,12.9,351.0,8.0,1.3,76.9,20.1,0.9,...,0.10,1.6,0,0,0,0.21,0.06,0.0,0,100.0
4,AR005,"Beras jagung kuning, kering, mentah",KZGMI-2001,10.8,358.0,5.5,0.1,82.7,10.0,0.9,...,0.10,4.1,NaN,641,NaN,0.12,0.08,1.0,3,100.0


In [ ]:
print("Baris invalid dihapus!")
print("Total baris valid:", len(tkpi_df))
display(tkpi_df.head())

Baris invalid dihapus!
Total baris valid: 1146


,KODE,NAMA BAHAN,SUMBER,AIR (g),ENERGI (Kal),PROTEIN (g),LEMAK (g),KH (g),SERAT (g),ABU (g),...,TEMBAGA (mg),SENG (mg),RETINOL (mcg),B-KAR (mcg),KAR-TOTAL (mcg),THIAMIN (mg),RIBOFLAVIN (mg),NIASIN (mg),VIT-C (mg),BDD (%)
0,AR001,"Beras giling, mentah",KZGMI-2001,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.10,0.5,0,0,0,0.20,0.08,2.6,0,100.0
1,AR002,"Beras giling var pelita, mentah",KZGPI- 1990,11.4,369.0,9.5,1.4,77.1,0.4,0.6,...,0.00,0.0,0,0,0,0.26,0.00,0.0,0,100.0
2,AR003,"Beras giling var rojolele, mentah",KZGPI- 1990,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.14,0.1,NaN,0,80,0.20,0.02,1.5,0,100.0
3,AR004,"Beras hitam, mentah",KZGMI-2001,12.9,351.0,8.0,1.3,76.9,20.1,0.9,...,0.10,1.6,0,0,0,0.21,0.06,0.0,0,100.0
4,AR005,"Beras jagung kuning, kering, mentah",KZGMI-2001,10.8,358.0,5.5,0.1,82.7,10.0,0.9,...,0.10,4.1,NaN,641,NaN,0.12,0.08,1.0,3,100.0


In [ ]:
print("1. Shape data:", tkpi_df.shape)
print("2. Kolom tersedia:")
print(tkpi_df.columns.tolist())

print("\n3. Sample 10 baris pertama:")
display(tkpi_df.head(10))

print("\n4. Info data (tipe & missing):")
print(tkpi_df.info())

print("\n5. Missing values per kolom:")
print(tkpi_df.isnull().sum())

print("\n6. Statistik deskriptif gizi utama:")
gizi_cols = ['ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'BESI (mg)', 'VIT-C (mg)', 'KALSIUM (mg)']
print(tkpi_df[gizi_cols].describe())

print("\n7. Bahan dengan protein tertinggi (top 10):")
print(tkpi_df.nlargest(10, 'PROTEIN (g)')[['NAMA BAHAN', 'PROTEIN (g)']])


1. Shape data: (1146, 25)
2. Kolom tersedia:
['KODE', 'NAMA BAHAN', 'SUMBER', 'AIR (g)', 'ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'KH (g)', 'SERAT (g)', 'ABU (g)', 'KALSIUM (mg)', 'FOSFOR (mg)', 'BESI (mg)', 'NATRIUM (mg)', 'KALIUM (mg)', 'TEMBAGA (mg)', 'SENG (mg)', 'RETINOL (mcg)', 'B-KAR (mcg)', 'KAR-TOTAL (mcg)', 'THIAMIN (mg)', 'RIBOFLAVIN (mg)', 'NIASIN (mg)', 'VIT-C (mg)', 'BDD (%)']

3. Sample 10 baris pertama:


,KODE,NAMA BAHAN,SUMBER,AIR (g),ENERGI (Kal),PROTEIN (g),LEMAK (g),KH (g),SERAT (g),ABU (g),...,TEMBAGA (mg),SENG (mg),RETINOL (mcg),B-KAR (mcg),KAR-TOTAL (mcg),THIAMIN (mg),RIBOFLAVIN (mg),NIASIN (mg),VIT-C (mg),BDD (%)
0,AR001,"Beras giling, mentah",KZGMI-2001,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.10,0.5,0,0,0,0.20,0.08,2.6,0,100.0
1,AR002,"Beras giling var pelita, mentah",KZGPI- 1990,11.4,369.0,9.5,1.4,77.1,0.4,0.6,...,0.00,0.0,0,0,0,0.26,0.00,0.0,0,100.0
2,AR003,"Beras giling var rojolele, mentah",KZGPI- 1990,12.0,357.0,8.4,1.7,77.1,0.2,0.8,...,0.14,0.1,NaN,0,80,0.20,0.02,1.5,0,100.0
3,AR004,"Beras hitam, mentah",KZGMI-2001,12.9,351.0,8.0,1.3,76.9,20.1,0.9,...,0.10,1.6,0,0,0,0.21,0.06,0.0,0,100.0
4,AR005,"Beras jagung kuning, kering, mentah",KZGMI-2001,10.8,358.0,5.5,0.1,82.7,10.0,0.9,...,0.10,4.1,NaN,641,NaN,0.12,0.08,1.0,3,100.0
5,AR006,"Beras jagung putih, kering, mentah",KZGMI-2001,22.5,307.0,4.8,0.1,71.8,10.0,0.8,...,0.10,3.5,NaN,301,NaN,0.15,0.07,0.9,0,100.0
6,AR007,"Beras ketan hitam tumbuk, mentah",KZGPI- 1990,13.7,360.0,8.0,2.3,74.5,1.0,1.5,...,0.28,2.2,0,0,0,0.24,0.10,2.0,0,100.0
7,AR008,"Beras ketan putih tumbuk, mentah",KZGPI- 1990,12.9,361.0,7.4,0.8,78.4,0.4,0.5,...,0.28,2.2,0,0,0,0.28,0.00,1.4,0,100.0
8,AR009,"Beras ladang, mentah",KZGMI-2001,9.8,376.0,7.5,3.8,78.0,5.9,0.9,...,0.10,1.4,0,0,NaN,0.20,0.20,5.1,0,100.0
9,AR010,"Beras menir, mentah",DABM-1964,12.0,362.0,7.7,4.4,73.0,0.2,0.2,...,0.10,0.5,0,0,NaN,0.55,0.00,1.9,0,100.0



4. Info data (tipe & missing):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1146 entries, 0 to 1145
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   KODE             1146 non-null   object 
 1   NAMA BAHAN       1146 non-null   object 
 2   SUMBER           1146 non-null   object 
 3   AIR (g)          1146 non-null   float64
 4   ENERGI (Kal)     1146 non-null   float64
 5   PROTEIN (g)      1145 non-null   float64
 6   LEMAK (g)        1145 non-null   object 
 7   KH (g)           1146 non-null   float64
 8   SERAT (g)        967 non-null    object 
 9   ABU (g)          1146 non-null   object 
 10  KALSIUM (mg)     1129 non-null   object 
 11  FOSFOR (mg)      1128 non-null   object 
 12  BESI (mg)        1125 non-null   object 
 13  NATRIUM (mg)     916 non-null    object 
 14  KALIUM (mg)      879 non-null    object 
 15  TEMBAGA (mg)     854 non-null    object 
 16  SENG (mg)        859 non-nul

In [ ]:
# Fix tipe data gizi (ubah object ke float, ganti non-numeric jadi 0)
gizi_cols = ['ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'KH (g)', 'SERAT (g)',
             'KALSIUM (mg)', 'FOSFOR (mg)', 'BESI (mg)', 'VIT-C (mg)']

for col in gizi_cols:
    tkpi_df[col] = pd.to_numeric(tkpi_df[col], errors='coerce')  # Non-numeric jadi NaN
    tkpi_df[col] = tkpi_df[col].fillna(0)  # NaN jadi 0

In [ ]:
print("\n8. Bahan dengan besi tertinggi (top 10):")
print(tkpi_df.nlargest(10, 'BESI (mg)')[['NAMA BAHAN', 'BESI (mg)']])

print("\n9. Jumlah varian tempe:")
print(len(tkpi_df[tkpi_df['NAMA BAHAN'].str.contains('tempe', case=False, na=False)]))

print("\n10. Jumlah varian ayam:")
print(len(tkpi_df[tkpi_df['NAMA BAHAN'].str.contains('ayam', case=False, na=False)]))


8. Bahan dengan besi tertinggi (top 10):
                     NAMA BAHAN  BESI (mg)
991   Teripang, dendeng, mentah       96.4
1142                     Terasi       78.5
1124          Daun salam, bubuk       44.1
317              Bungkil kelapa       41.5
349    Oncom kacang tanah pepes       34.4
1076                 Lemak ikan       32.7
1107     Teh melati daun kering       31.6
316        Bungkil kacang tanah       30.7
1104               Sirup pirous       29.5
334               Keripik oncom       27.0

9. Jumlah varian tempe:
20

10. Jumlah varian ayam:
33


In [ ]:
tempe_df = tkpi_df[tkpi_df['NAMA BAHAN'].str.contains('tempe', case=False, na=False)]
tempe_df[['NAMA BAHAN', 'ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'BESI (mg)']]


,NAMA BAHAN,ENERGI (Kal),PROTEIN (g),LEMAK (g),BESI (mg)
335,Keripik tempe,581.0,12.1,40.6,6.9
336,Keripik tempe abadi besar,556.0,15.8,37.1,5.4
337,Keripik tempe abadi murni,542.0,40.3,42.4,5.2
338,Keripik tempe abadi sedang,510.0,12.3,27.9,8.4
339,Keripik tempe abadi telur,529.0,20.8,31.7,5.4
340,Kerupik tempe abadi prima,540.0,16.7,33.9,3.2
365,Tempe bongkrek,119.0,4.4,3.5,2.6
366,Tempe gembus P3G,73.0,5.7,1.3,1.5
367,Tempe gembus yogya,76.0,6.8,0.7,16.5
368,Tempe kacang babi,139.0,12.5,0.8,2.6


In [ ]:
import pandas as pd

# Filter ayam murni (tanpa bayam & pisang ayam)
ayam_df = tkpi_df[
    tkpi_df['NAMA BAHAN'].str.contains(r'\bayam\b', case=False, regex=True, na=False) &
    ~tkpi_df['NAMA BAHAN'].str.contains(r'\bbayam\b|\bpisang ayam\b', case=False, regex=True, na=False)
].reset_index(drop=True)

print("Jumlah varian ayam (tanpa bayam & pisang ayam):", len(ayam_df))
display(ayam_df[['NAMA BAHAN', 'ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'BESI (mg)']])

Jumlah varian ayam (tanpa bayam & pisang ayam): 25


,NAMA BAHAN,ENERGI (Kal),PROTEIN (g),LEMAK (g),BESI (mg)
0,Mie ayam,102.0,6.2,3.9,1.8
1,"Ayam, daging, segar",298.0,18.2,25.0,1.5
2,"Ayam, dideh/darah, segar",75.0,13.8,1.9,1.3
3,"Ayam, hati, segar",261.0,27.4,16.1,15.8
4,"Ayam, ampela, goreng",270.0,32.3,11.2,4.9
5,"Ayam, usus, goreng",473.0,45.2,26.3,8.4
6,"Ayam goreng church texas, dada",338.0,35.2,20.6,5.6
7,"Ayam goreng church, texas paha",287.0,31.0,15.7,4.1
8,"Ayam goreng church texas, sayap",295.0,34.0,16.0,3.0
9,"Ayam goreng kalasan, paha",275.0,37.4,12.2,5.8


In [ ]:
import pandas as pd

# Filter ayam murni (tanpa bayam & pisang ayam)
ayam_df = tkpi_df[
    tkpi_df['NAMA BAHAN'].str.contains(r'\bayam\b', case=False, regex=True, na=False) &
    ~tkpi_df['NAMA BAHAN'].str.contains(r'\bbayam\b|\bpisang ayam\b', case=False, regex=True, na=False)
].reset_index(drop=True)

print("Jumlah varian ayam (tanpa bayam & pisang ayam):", len(ayam_df))
display(ayam_df[['NAMA BAHAN', 'ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'BESI (mg)']])


Jumlah varian ayam (tanpa bayam & pisang ayam): 25


,NAMA BAHAN,ENERGI (Kal),PROTEIN (g),LEMAK (g),BESI (mg)
0,Mie ayam,102.0,6.2,3.9,1.8
1,"Ayam, daging, segar",298.0,18.2,25.0,1.5
2,"Ayam, dideh/darah, segar",75.0,13.8,1.9,1.3
3,"Ayam, hati, segar",261.0,27.4,16.1,15.8
4,"Ayam, ampela, goreng",270.0,32.3,11.2,4.9
5,"Ayam, usus, goreng",473.0,45.2,26.3,8.4
6,"Ayam goreng church texas, dada",338.0,35.2,20.6,5.6
7,"Ayam goreng church, texas paha",287.0,31.0,15.7,4.1
8,"Ayam goreng church texas, sayap",295.0,34.0,16.0,3.0
9,"Ayam goreng kalasan, paha",275.0,37.4,12.2,5.8


In [ ]:
import pandas as pd

# Asumsikan tkpi_df sudah ada dari code sebelumnya
# Jika belum, load ulang
tkpi_df = pd.read_excel(TKPI_PATH, sheet_name="Komposisi Pangan", skiprows=3)
tkpi_df = tkpi_df.dropna(how='all')
tkpi_df.columns = ['KODE','NAMA BAHAN','SUMBER','AIR (g)','ENERGI (Kal)','PROTEIN (g)','LEMAK (g)','KH (g)','SERAT (g)','ABU (g)','KALSIUM (mg)','FOSFOR (mg)','BESI (mg)','NATRIUM (mg)','KALIUM (mg)','TEMBAGA (mg)','SENG (mg)','RETINOL (mcg)','B-KAR (mcg)','KAR-TOTAL (mcg)','THIAMIN (mg)','RIBOFLAVIN (mg)','NIASIN (mg)','VIT-C (mg)','BDD (%)']

# Fix tipe & missing
gizi_cols = ['ENERGI (Kal)', 'PROTEIN (g)', 'LEMAK (g)', 'KH (g)', 'SERAT (g)', 'KALSIUM (mg)', 'FOSFOR (mg)', 'BESI (mg)', 'NATRIUM (mg)', 'VIT-C (mg)', 'KAR-TOTAL (mcg)']
for col in gizi_cols:
    tkpi_df[col] = pd.to_numeric(tkpi_df[col], errors='coerce').fillna(0)

# Hapus baris invalid
tkpi_df = tkpi_df.dropna(subset=['NAMA BAHAN'])
tkpi_df = tkpi_df[tkpi_df['NAMA BAHAN'].str.strip() != "TUNGGAL/SINGLE"]

# Buat narasi & metadata
narasi_list = []
metadata_list = []

for _, row in tkpi_df.iterrows():
    nama = row['NAMA BAHAN'].strip().title()
    kode = row['KODE']
    sumber = row['SUMBER']
    energi = row['ENERGI (Kal)']
    protein = row['PROTEIN (g)']
    lemak = row['LEMAK (g)']
    kh = row['KH (g)']
    serat = row['SERAT (g)']
    besi = row['BESI (mg)']
    kalsium = row['KALSIUM (mg)']
    vit_c = row['VIT-C (mg)']
    vit_a = row['KAR-TOTAL (mcg)']

    # Extra keyword
    extra = ""
    if protein > 30:
        extra += "Sangat tinggi protein, cocok untuk olahraga dan pembentukan otot. "
    elif protein > 15:
        extra += "Tinggi protein. "
    if besi > 10:
        extra += "Sangat kaya zat besi, baik untuk anemia. "
    elif besi > 5:
        extra += "Kaya zat besi. "
    if lemak < 5:
        extra += "Rendah lemak. "
    if energi < 100:
        extra += "Rendah kalori, cocok untuk diet. "
    if serat > 5:
        extra += "Tinggi serat, baik untuk pencernaan. "
    if kalsium > 200:
        extra += "Kaya kalsium, baik untuk tulang dan anak. "
    if vit_c > 50:
        extra += "Kaya vitamin C, baik untuk imun. "

    # Narasi
    narasi = f"Bahan: {nama}\n{extra}\nKandungan gizi per 100 gram:\n- Energi: {energi} kkal\n- Protein: {protein} g\n- Lemak: {lemak} g\n- Karbohidrat: {kh} g\n- Serat: {serat} g\n- Zat Besi: {besi} mg\n- Kalsium: {kalsium} mg\n- Vitamin C: {vit_c} mg\n- Vitamin A: {vit_a} mcg"

    narasi_list.append(narasi)

    # Metadata lengkap (termasuk KODE & SUMBER)
    metadata_list.append({
        "kode": kode,
        "sumber": sumber,
        "nama": nama,
        "kalori": energi,
        "protein": protein,
        "lemak": lemak,
        "karbohidrat": kh,
        "serat": serat,
        "besi": besi,
        "kalsium": kalsium,
        "vit_c": vit_c,
        "vit_a": vit_a
    })

print(f"Narasi TKPI selesai! Total: {len(narasi_list)}")
print("\nContoh narasi pertama:")
print(narasi_list[0])
print("\nContoh metadata pertama:")
print(metadata_list[0])

Narasi TKPI selesai! Total: 1146

Contoh narasi pertama:
Bahan: Beras Giling, Mentah
Rendah lemak. 
Kandungan gizi per 100 gram:
- Energi: 357.0 kkal
- Protein: 8.4 g
- Lemak: 1.7 g
- Karbohidrat: 77.1 g
- Serat: 0.2 g
- Zat Besi: 1.8 mg
- Kalsium: 147.0 mg
- Vitamin C: 0.0 mg
- Vitamin A: 0.0 mcg

Contoh metadata pertama:
{'kode': 'AR001', 'sumber': 'KZGMI-2001', 'nama': 'Beras Giling, Mentah', 'kalori': 357.0, 'protein': 8.4, 'lemak': 1.7, 'karbohidrat': 77.1, 'serat': 0.2, 'besi': 1.8, 'kalsium': 147.0, 'vit_c': 0.0, 'vit_a': 0.0}


In [ ]:
import json

# Path folder processed di Drive
PROCESSED_PATH = "/content/drive/MyDrive/skripsian/data/processed"
# Buat folder kalau belum ada
import os
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Simpan narasi
narasi_file = f"{PROCESSED_PATH}/narasi_tkpi.json"
with open(narasi_file, 'w', encoding='utf-8') as f:
    json.dump(narasi_list, f, ensure_ascii=False, indent=4)

# Simpan metadata
metadata_file = f"{PROCESSED_PATH}/metadata_tkpi.json"
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=4)

print("Narasi & metadata TKPI berhasil disimpan di Drive!")
print(f"Narasi: {narasi_file}")
print(f"Metadata: {metadata_file}")
print(f"Total narasi: {len(narasi_list)}")

Narasi & metadata TKPI berhasil disimpan di Drive!
Narasi: /content/drive/MyDrive/skripsian/data/processed/narasi_tkpi.json
Metadata: /content/drive/MyDrive/skripsian/data/processed/metadata_tkpi.json
Total narasi: 1146


### Code Text + Numeric + Hybrid Embedding

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
import chromadb

In [ ]:
# 1. Load model text embedding
model = SentenceTransformer('intfloat/multilingual-e5-large')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
# 2. Text embedding (passage prefix untuk optimal retrieval)
text_embeddings = model.encode(['passage: ' + n for n in narasi_list], normalize_embeddings=True)

print("Text embedding selesai! Shape:", text_embeddings.shape)

Text embedding selesai! Shape: (1146, 1024)


In [ ]:
# 3. Numeric embedding
# Kolom numeric (10 kolom prioritas)
numeric_cols = ['kalori', 'protein', 'lemak', 'karbohidrat', 'serat', 'besi', 'kalsium', 'vit_c', 'vit_a', 'natrium_mg']  # adjust kalau ada

# Buat array numeric dari metadata_list
nutrition_data = np.array([
    [
        m['kalori'],
        m['protein'],
        m['lemak'],
        m['karbohidrat'],
        m['serat'],
        m['besi'],
        m['kalsium'],
        m['vit_c'],
        m['vit_a'],
        m.get('natrium_mg', 0)  # kalau ada
    ] for m in metadata_list
])

# Normalisasi & reduce dimensi ke 10
nutrition_scaled = MinMaxScaler().fit_transform(nutrition_data)
nutrition_reduced = TruncatedSVD(n_components=10, random_state=42).fit_transform(nutrition_scaled)

print("Numeric embedding selesai! Shape:", nutrition_reduced.shape)  # (n, 10)

Numeric embedding selesai! Shape: (1146, 10)


In [ ]:
# 4. Hybrid embedding (bobot numeric 1.0 — teks dominan)
hybrid_embeddings = np.hstack([text_embeddings, nutrition_reduced * 1.0])

print("Hybrid embedding selesai! Shape:", hybrid_embeddings.shape)  # (n, 1034)

Hybrid embedding selesai! Shape: (1146, 1034)


In [ ]:
# 5. Ingest ke ChromaDB baru (di folder skripsian)
CHROMA_PATH = "/content/drive/MyDrive/skripsian/chroma_db/chroma_tkpi_full"
client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection("tkpi_full")
except:
    pass

collection = client.create_collection("tkpi_full")

collection.add(
    ids=[f"tkpi_{i}" for i in range(len(narasi_list))],
    documents=narasi_list,
    embeddings=hybrid_embeddings.tolist(),
    metadatas=metadata_list
)

print("Ingest ke ChromaDB selesai!")
print(f"Collection 'tkpi_full' siap dengan {len(narasi_list)} bahan dari TKPI 2019")

Ingest ke ChromaDB selesai!
Collection 'tkpi_full' siap dengan 1146 bahan dari TKPI 2019


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb

# Load model embed
model = SentenceTransformer('intfloat/multilingual-e5-large')

# Load collection TKPI full
CHROMA_PATH = "/content/drive/MyDrive/skripsian/chroma_db/chroma_tkpi_full"
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection("tkpi_full")

# Fungsi test retrieval
def test_retrieval(query, k=5):
    # Query embedding hybrid (bobot numeric 1.0)
    q_embed = model.encode(['query: ' + query], normalize_embeddings=True)
    q_hybrid = np.hstack([q_embed, np.zeros((1, 10)) * 1.0])

    results = collection.query(
        query_embeddings=q_hybrid.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    print(f"\nQuery: '{query}'")
    print("-" * 80)
    for i in range(k):
        nama = results['metadatas'][0][i]['nama']
        protein = results['metadatas'][0][i]['protein']
        besi = results['metadatas'][0][i]['besi']
        distance = results['distances'][0][i]
        print(f"Rank {i+1} (distance {distance:.4f}): {nama}")
        print(f"   Protein: {protein} g | Besi: {besi} mg")
        print(f"   Narasi preview: {results['documents'][0][i][:200]}...\n")

# Test berbagai query
queries = [
    "Sangat tinggi protein untuk olahraga",
    "kaya zat besi untuk anemia",
    "rendah kalori untuk diet",
    "tinggi serat untuk pencernaan",
    "dada ayam goreng",
    "hati ayam",
    "tempe gembus",
    "keripik tempe renyah",
    "berapa protein dada ayam goreng",
    "menu sarapan sehat anak"
]

for q in queries:
    test_retrieval(q)


Query: 'Sangat tinggi protein untuk olahraga'
--------------------------------------------------------------------------------
Rank 1 (distance 0.3332): Olah-Olah
   Protein: 2.1 g | Besi: 1.2 mg
   Narasi preview: Bahan: Olah-Olah
Rendah kalori, cocok untuk diet. 
Kandungan gizi per 100 gram:
- Energi: 9.0 kkal
- Protein: 2.1 g
- Lemak: 6.0 g
- Karbohidrat: 7.1 g
- Serat: 4.4 g
- Zat Besi: 1.2 mg
- Kalsium: 26....

Rank 2 (distance 0.3454): Lontar, Segar
   Protein: 0.4 g | Besi: 0.5 mg
   Narasi preview: Bahan: Lontar, Segar
Rendah lemak. Rendah kalori, cocok untuk diet. 
Kandungan gizi per 100 gram:
- Energi: 27.0 kkal
- Protein: 0.4 g
- Lemak: 0.2 g
- Karbohidrat: 6.0 g
- Serat: 1.6 g
- Zat Besi: 0....

Rank 3 (distance 0.3480): Sawi Putih / Pecai,
 Segar
   Protein: 1.0 g | Besi: 1.1 mg
   Narasi preview: Bahan: Sawi Putih / Pecai,
 Segar
Rendah lemak. Rendah kalori, cocok untuk diet. 
Kandungan gizi per 100 gram:
- Energi: 9.0 kkal
- Protein: 1.0 g
- Lemak: 0.1 g
- Karbohidrat: 

### Build ulang Re-Ingest

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load model
model = SentenceTransformer('intfloat/multilingual-e5-large')

# Embedding hanya teks (dengan prefix passage untuk optimal)
print("Generating embeddings...")
text_embeddings = model.encode(['passage: ' + n for n in narasi_list],
                               normalize_embeddings=True,
                               show_progress_bar=True)

# Path ChromaDB
CHROMA_PATH = "/content/drive/MyDrive/skripsian/chroma_db/chroma_tkpi_full"

# Buat client baru
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Hapus collection lama jika masih ada (safety)
try:
    client.delete_collection("tkpi_full")
except:
    pass

# Buat collection baru
collection = client.create_collection(name="tkpi_full")

# Ingest dengan pure text embedding
print("Ingesting to ChromaDB...")
collection.add(
    ids=[f"tkpi_{i}" for i in range(len(narasi_list))],
    documents=narasi_list,
    embeddings=text_embeddings.tolist(),   # 1024 dim saja
    metadatas=metadata_list
)

print("Ingest ulang SELESAI!")
print(f"Total item: {collection.count()}")
print(f"Dimensi embedding: {len(text_embeddings[0])}")  # Harusnya 1024

Generating embeddings...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Ingesting to ChromaDB...
Ingest ulang SELESAI!
Total item: 1146
Dimensi embedding: 1024


In [ ]:
import re

def parse_query(query):
    query_lower = query.lower()

    if re.search(r'(tinggi|sangat tinggi|kaya|banyak|penambah)', query_lower):
        if 'protein' in query_lower:
            return {'col': 'protein', 'order': 'desc'}
        elif any(x in query_lower for x in ['zat besi', 'besi', 'iron', 'darah']):
            return {'col': 'besi', 'order': 'desc'}
        elif 'serat' in query_lower:
            return {'col': 'serat', 'order': 'desc'}
        elif any(x in query_lower for x in ['kalsium', 'tulang']):
            return {'col': 'kalsium', 'order': 'desc'}
        elif any(x in query_lower for x in ['vitamin c', 'vit c', 'daya tahan']):
            return {'col': 'vit_c', 'order': 'desc'}
        elif any(x in query_lower for x in ['vitamin a', 'vit a', 'mata']):
            return {'col': 'vit_a', 'order': 'desc'}
        elif any(x in query_lower for x in ['energi', 'kalori tinggi']):
            return {'col': 'kalori', 'order': 'desc'}

    elif re.search(r'(rendah|kecil|sedikit|diet)', query_lower):
        if 'kalori' in query_lower:
            return {'col': 'kalori', 'order': 'asc'}
        elif 'lemak' in query_lower:
            return {'col': 'lemak', 'order': 'asc'}
        elif any(x in query_lower for x in ['natrium', 'garam']):
            return {'col': 'natrium_mg', 'order': 'asc'}
        elif any(x in query_lower for x in ['karbohidrat', 'diabetes']):
            return {'col': 'karbohidrat', 'order': 'asc'}

    return None

    # --- 2. Intent Detection (rule-based) ---
INTENT_NUTRITION = 'nutrition'
INTENT_FOOD_NAME = 'food_name'
INTENT_MEAL_CONTEXT = 'meal_context'

def detect_intent(query):
    query_lower = query.lower()

    # Nutrition intent
    nutrition_keywords = ['protein', 'zat besi', 'besi', 'kalori', 'serat', 'kalsium', 'vitamin', 'vit', 'energi', 'lemak', 'natrium', 'karbohidrat', 'kaya', 'tinggi', 'rendah']
    if any(k in query_lower for k in nutrition_keywords):
        return INTENT_NUTRITION

    # Meal / context intent
    context_keywords = ['sarapan', 'makan malam', 'cemilan', 'ngemil', 'diet', 'diabetes', 'hamil', 'anak', 'sekolah', 'olahraga', 'atlet', 'penambah darah', 'mengenyangkan', 'menu']
    if any(k in query_lower for k in context_keywords):
        return INTENT_MEAL_CONTEXT

    # Default: food name intent
    return INTENT_FOOD_NAME

# --- 3. Hybrid Retrieval dengan Intent Routing ---
def hybrid_retrieval(query, k=5, candidate_k=50):
    intent = detect_intent(query)

    # Embedding query
    q_embed = model.encode(['query: ' + query], normalize_embeddings=True)

    # Semantic retrieval top candidates
    results = collection.query(
        query_embeddings=q_embed.tolist(),
        n_results=candidate_k,
        include=["documents", "metadatas", "distances"]
    )

    candidates = results['metadatas'][0] if results['metadatas'] else []
    distances = results['distances'][0] if results['distances'] else [0] * len(candidates)

    # Mapping index untuk distance
    def get_distance(idx):
        try:
            return distances[candidates.index(results['metadatas'][0][idx])]
        except:
            return 1.0

    # Routing berdasarkan intent
    if intent == INTENT_NUTRITION:
        criteria = parse_query(query)
        if criteria and candidates:
            sorted_cands = sorted(
                candidates,
                key=lambda x: float(x.get(criteria['col'], 0) or 0),
                reverse=(criteria['order'] == 'desc')
            )
            return sorted_cands[:k]
        else:
            return candidates[:k]  # fallback semantic

    elif intent == INTENT_FOOD_NAME:
        # Extract kata kunci utama (protein source)
        main_ingredients = re.findall(r'\b(ayam|tempe|ikan|daging|sapi|kambing|tuna|telur|tahu|kacang)\b', query_lower := query.lower())

        if main_ingredients:
            # Boost item yang punya kata utama di nama
            sorted_cands = sorted(
                candidates,
                key=lambda x: (
                    -sum(ing in x['nama'].lower() for ing in main_ingredients),  # lebih banyak match = lebih tinggi
                    get_distance(candidates.index(x))  # tie-breaker: semantic distance
                )
            )
        else:
            # Pure semantic
            sorted_cands = sorted(candidates, key=lambda x: get_distance(candidates.index(x)))

        return sorted_cands[:k]

    elif intent == INTENT_MEAL_CONTEXT:
        # Heuristic filter: hindari makanan ekstrem untuk konteks anak/sarapan/diet
        exclude_words = ['mentah', 'kering', 'bubuk', 'dendeng', 'teripang', 'terasi']
        filtered = [c for c in candidates if not any(word in c['nama'].lower() for word in exclude_words)]

        if not filtered:  # kalau terlalu ketat, fallback
            filtered = candidates

        # Jika ada kriteria nutrisi, rerank
        criteria = parse_query(query)
        if criteria:
            sorted_cands = sorted(
                filtered,
                key=lambda x: float(x.get(criteria['col'], 0) or 0),
                reverse=(criteria['order'] == 'desc')
            )
        else:
            sorted_cands = sorted(filtered, key=lambda x: get_distance(candidates.index(x)))

        return sorted_cands[:k]

    return candidates[:k]
# --- 4. Test 50 Query ---
test_queries = [
    "Berapa kandungan gizi lengkap beras giling mentah per 100 gram?",
    "Apa saja zat gizi dalam beras hitam mentah, termasuk energi dan protein?",
    "Kandungan kalori, protein, dan lemak di beras jagung kuning kering mentah per 100 gram?",
    "Berapa besi dan kalsium dalam beras ketan hitam tumbuk mentah?",
    "Kandungan gizi lengkap tempe kedelai murni goreng per 100 gram?",
    "Apa protein dan serat di tempe pasar goreng?",
    "Kandungan energi, lemak, dan karbohidrat di tempe lamtoro?",
    "Berapa vitamin A dan besi dalam ayam goreng Kentucky dada?",
    "Kandungan gizi lengkap ayam goreng Church Texas dada per 100 gram?",
    "Apa kalori dan protein di ayam hati segar?",
    "Kandungan besi, kalsium, dan vitamin C di ikan mujahir segar?",
    "Berapa protein dan lemak di ikan teri kering tawar mentah?",
    "Kandungan gizi lengkap ikan sale lais mentah per 100 gram?",
    "Apa energi dan serat di jagung muda kuning mentah?",
    "Kandungan kalori, protein, dan besi di kacang kedelai goreng?",
    "Berapa kalsium dan vitamin A di susu bubuk?",
    "Kandungan gizi lengkap kecap per 100 gram?",
    "Apa lemak dan natrium di terasi?",
    "Kandungan besi dan protein di oncom kacang tanah pepes?",
    "Berapa kalori dan serat di rebung segar?",
    "Kandungan gizi lengkap daun singkong segar per 100 gram?",
    "Apa protein dan vitamin C di jambu biji segar?",
    "Kandungan energi, lemak, dan kalsium di keju?",
    "Berapa besi dan vitamin A di hati babi segar?",
    "Kandungan gizi lengkap nasi per 100 gram?",
    "Apa serat dan karbohidrat di beras merah nasi?",
    "Kandungan protein, besi, dan kalium di kacang bogor kering?",
    "Berapa kalori dan lemak di martabak india?",
    "Kandungan gizi lengkap mie ayam per 100 gram?",
    "Apa vitamin C dan besi di cabai merah segar?",
    "Kandungan energi, protein, dan serat di durian segar?",
    "Berapa kalsium dan besi di ikan bandeng presto masakan?",
    "Kandungan gizi lengkap sop buntut masakan per 100 gram?",
    "Apa lemak dan natrium di saos tomat?",
    "Kandungan protein dan vitamin A di daun mangkokan segar?",
    "Berapa besi dan serat di kacang endel biji kering?",
    "Kandungan gizi lengkap petis udang pasta per 100 gram?",
    "Apa kalori dan kalsium di air kelapa muda segar?",
    "Kandungan energi, lemak, dan karbohidrat di biskuit?",
    "Berapa protein dan besi di daging sapi kornet?",
    "Kandungan gizi lengkap gado-gado per 100 gram?",
    "Apa serat dan vitamin C di salak medan segar?",
    "Kandungan besi, kalsium, dan vitamin A di genjer segar?",
    "Berapa kalori dan protein di bakwan?",
    "Kandungan gizi lengkap rendang sapi masakan per 100 gram?",
    "Apa lemak dan natrium di bekasam?",
    "Kandungan energi, serat, dan besi di cabai merah kering?",
    "Berapa vitamin A dan kalsium di andaliman segar?",
    "Kandungan gizi lengkap kwaci per 100 gram?",
    "Apa protein, besi, dan kalium di wijen mentah?"
]

print(f"Memulai pengujian {len(test_queries)} query dengan Intent Detection + Routing...\n")
print("="*120)

for idx, query in enumerate(test_queries, 1):
    intent = detect_intent(query)
    print(f"{idx:2}. QUERY: \"{query}\"")
    print(f"     → Detected Intent: {intent.upper()}")
    print("-" * 100)

    results = hybrid_retrieval(query, k=5)

    for rank, item in enumerate(results, 1):
        nama = item['nama']
        kalori = item.get('kalori', 'N/A')
        protein = item.get('protein', 'N/A')
        lemak = item.get('lemak', 'N/A')
        karbohidrat = item.get('karbohidrat', 'N/A')
        serat = item.get('serat', 'N/A')
        kalsium = item.get('kalsium', 'N/A')
        besi = item.get('besi', 'N/A')
        vit_a = item.get('vit_a', 'N/A')
        vit_c = item.get('vit_c', 'N/A')

        print(f"   {rank}. {nama}")
        print(f"      Kalori: {kalori}kkal |  Protein: {protein}g | Lemak: {lemak}g | Karbohidrat: {karbohidrat}g | Serat: {serat}g | Kalsium: {kalsium}mg | Besi: {besi}mg | Retinol: {vit_a}mcg | Vit C: {vit_c}mg")

    print("\n")

Memulai pengujian 50 query dengan Intent Detection + Routing...

 1. QUERY: "Berapa kandungan gizi lengkap beras giling mentah per 100 gram?"
     → Detected Intent: FOOD_NAME
----------------------------------------------------------------------------------------------------
   1. Beras Giling, Mentah
      Kalori: 357.0kkal |  Protein: 8.4g | Lemak: 1.7g | Karbohidrat: 77.1g | Serat: 0.2g | Kalsium: 147.0mg | Besi: 1.8mg | Retinol: 0.0mcg | Vit C: 0.0mg
   2. Beras Giling Var Pelita, Mentah
      Kalori: 369.0kkal |  Protein: 9.5g | Lemak: 1.4g | Karbohidrat: 77.1g | Serat: 0.4g | Kalsium: 68.0mg | Besi: 1.4mg | Retinol: 0.0mcg | Vit C: 0.0mg
   3. Beras, Tepung, Mentah
      Kalori: 353.0kkal |  Protein: 7.0g | Lemak: 0.5g | Karbohidrat: 80.0g | Serat: 2.4g | Kalsium: 5.0mg | Besi: 0.8mg | Retinol: 0.0mcg | Vit C: 0.0mg
   4. Beras Menir, Mentah
      Kalori: 362.0kkal |  Protein: 7.7g | Lemak: 4.4g | Karbohidrat: 73.0g | Serat: 0.2g | Kalsium: 22.0mg | Besi: 3.7mg | Retinol: 0.0mcg

In [ ]:
import re

# Enum sederhana untuk intent
INTENT_NUTRITION = 'nutrition'  # fokus gizi (tinggi protein, rendah kalori)
INTENT_FOOD_NAME = 'food_name'  # nama spesifik (dada ayam goreng)
INTENT_MEAL_CONTEXT = 'meal_context'  # konteks (sarapan anak, diet diabetes)

def detect_intent(query):
    query_lower = query.lower()

    # Rule 1: Nutrition intent (ada kata gizi seperti protein, besi, kalori)
    nutrition_keywords = ['protein', 'zat besi', 'besi', 'kalori', 'serat', 'kalsium', 'vit', 'vitamin', 'energi', 'lemak', 'natrium', 'karbohidrat', 'kaya', 'tinggi', 'rendah']
    if any(k in query_lower for k in nutrition_keywords):
        return INTENT_NUTRITION

    # Rule 2: Meal/context intent (ada kata konteks seperti sarapan, anak, diet, ibu hamil)
    context_keywords = ['sarapan', 'makan malam', 'cemilan', 'ngemil', 'diet', 'diabetes', 'hamil', 'anak', 'sekolah', 'olahraga', 'atlet', 'penambah darah', 'mengenyangkan']
    if any(k in query_lower for k in context_keywords):
        return INTENT_MEAL_CONTEXT

    # Rule 3: Food name intent (default jika nama makanan spesifik, tanpa keyword gizi/konteks)
    return INTENT_FOOD_NAME

def hybrid_retrieval(query, k=5, candidate_k=50):
    intent = detect_intent(query)
    print(f"Detected intent: {intent}")  # Debug untuk skripsi

    q_embed = model.encode(['query: ' + query], normalize_embeddings=True)

    # Retrieval dasar (semantic top-50)
    results = collection.query(
        query_embeddings=q_embed.tolist(),
        n_results=candidate_k,
        include=["documents", "metadatas", "distances"]
    )

    candidates = results['metadatas'][0] if results['metadatas'] else []

    # Routing berdasarkan intent
    if intent == INTENT_NUTRITION:
        # Standard: Parse criteria + rerank numeric
        criteria = parse_query(query)
        if criteria:
            sorted_cands = sorted(
                candidates,
                key=lambda x: float(x.get(criteria['col'], 0) or 0),
                reverse=(criteria['order'] == 'desc')
            )
            return sorted_cands[:k]

    elif intent == INTENT_FOOD_NAME:
        # Boost keyword match: Sort by distance (semantic) + keyword di nama
        protein_keywords = re.findall(r'\b(ayam|tempe|ikan|daging|sapi|kambing)\b', query.lower())  # Extract protein utama
        if protein_keywords:
            sorted_cands = sorted(
                candidates,
                key=lambda x: (-sum(k in x['nama'].lower() for k in protein_keywords),  # Boost utama
                               results['distances'][0][candidates.index(x)])  # Tie-breaker distance
            )
        else:
            sorted_cands = sorted(candidates, key=lambda x: results['distances'][0][candidates.index(x)])
        return sorted_cands[:k]

    elif intent == INTENT_MEAL_CONTEXT:
        # Heuristic filter: Exclude "mentah", "kering", "bubuk" jika konteks anak/sarapan
        filtered_cands = [c for c in candidates if all(w not in c['nama'].lower() for w in ['mentah', 'kering', 'bubuk', 'dendeng'])]
        if not filtered_cands:
            filtered_cands = candidates  # Fallback jika terlalu ketat

        # Lalu rerank numeric jika ada criteria
        criteria = parse_query(query)
        if criteria:
            sorted_cands = sorted(
                filtered_cands,
                key=lambda x: float(x.get(criteria['col'], 0) or 0),
                reverse=(criteria['order'] == 'desc')
            )
        else:
            sorted_cands = sorted(filtered_cands, key=lambda x: results['distances'][0][candidates.index(x)])
        return sorted_cands[:k]

    # Fallback jika None
    return candidates[:k]

In [ ]:
test_cases = [
    {
        "query": "Berapa kandungan gizi lengkap beras giling mentah per 100 gram?",
        "expected": "beras giling"
    },
    {
        "query": "Apa saja zat gizi dalam beras hitam mentah, termasuk energi dan protein?",
        "expected": "beras hitam"
    },
    {
        "query": "Kandungan kalori, protein, dan lemak di beras jagung kuning kering mentah per 100 gram?",
        "expected": "beras jagung kuning"
    },
    {
        "query": "Berapa besi dan kalsium dalam beras ketan hitam tumbuk mentah?",
        "expected": "beras ketan hitam"
    },
    {
        "query": "Kandungan gizi lengkap tempe kedelai murni goreng per 100 gram?",
        "expected": "tempe kedelai murni"
    },
    {
        "query": "Apa protein dan serat di tempe pasar goreng?",
        "expected": "tempe pasar"
    },
    {
        "query": "Kandungan energi, lemak, dan karbohidrat di tempe lamtoro?",
        "expected": "tempe lamtoro"
    },
    {
        "query": "Berapa vitamin A dan besi dalam ayam goreng Kentucky dada?",
        "expected": "ayam goreng kentucky, dada"
    },
    {
        "query": "Kandungan gizi lengkap ayam goreng Church Texas dada per 100 gram?",
        "expected": "ayam goreng church texas, dada"
    },
    {
        "query": "Apa kalori dan protein di ayam hati segar?",
        "expected": "ayam hati"
    },
    {
        "query": "Kandungan besi, kalsium, dan vitamin C di ikan mujahir segar?",
        "expected": "ikan mujahir"
    },
    {
        "query": "Berapa protein dan lemak di ikan teri kering tawar mentah?",
        "expected": "ikan teri"
    },
    {
        "query": "Kandungan gizi lengkap ikan sale lais mentah per 100 gram?",
        "expected": "ikan sale lais"
    },
    {
        "query": "Apa energi dan serat di jagung muda kuning mentah?",
        "expected": "jagung muda kuning"
    },
    {
        "query": "Kandungan kalori, protein, dan besi di kacang kedelai goreng?",
        "expected": "kacang kedelai"
    },
    {
        "query": "Berapa kalsium dan vitamin A di susu bubuk?",
        "expected": "susu bubuk"
    },
    {
        "query": "Kandungan gizi lengkap kecap per 100 gram?",
        "expected": "kecap"
    },
    {
        "query": "Apa lemak dan natrium di terasi?",
        "expected": "terasi"
    },
    {
        "query": "Kandungan besi dan protein di oncom kacang tanah pepes?",
        "expected": "oncom kacang tanah"
    },
    {
        "query": "Berapa kalori dan serat di rebung segar?",
        "expected": "rebung"
    },
    {
        "query": "Kandungan gizi lengkap daun singkong segar per 100 gram?",
        "expected": "daun singkong"
    },
    {
        "query": "Apa protein dan vitamin C di jambu biji segar?",
        "expected": "jambu biji"
    },
    {
        "query": "Kandungan energi, lemak, dan kalsium di keju?",
        "expected": "keju"
    },
    {
        "query": "Berapa besi dan vitamin A di hati babi segar?",
        "expected": "hati babi"
    },
    {
        "query": "Kandungan gizi lengkap nasi per 100 gram?",
        "expected": "nasi"
    },
    {
        "query": "Apa serat dan karbohidrat di beras merah nasi?",
        "expected": "beras merah"
    },
    {
        "query": "Kandungan protein, besi, dan kalium di kacang bogor kering?",
        "expected": "kacang bogor"
    },
    {
        "query": "Berapa kalori dan lemak di martabak india?",
        "expected": "martabak india"
    },
    {
        "query": "Kandungan gizi lengkap mie ayam per 100 gram?",
        "expected": "mie ayam"
    },
    {
        "query": "Apa vitamin C dan besi di cabai merah segar?",
        "expected": "cabai merah"
    },
    {
        "query": "Kandungan energi, protein, dan serat di durian segar?",
        "expected": "durian"
    },
    {
        "query": "Berapa kalsium dan besi di ikan bandeng presto masakan?",
        "expected": "ikan bandeng presto"
    },
    {
        "query": "Kandungan gizi lengkap sop buntut masakan per 100 gram?",
        "expected": "sop buntut"
    },
    {
        "query": "Apa lemak dan natrium di saos tomat?",
        "expected": "saos tomat"
    },
    {
        "query": "Kandungan protein dan vitamin A di daun mangkokan segar?",
        "expected": "daun mangkokan"
    },
    {
        "query": "Berapa besi dan serat di kacang endel biji kering?",
        "expected": "kacang endel"
    },
    {
        "query": "Kandungan gizi lengkap petis udang pasta per 100 gram?",
        "expected": "petis udang"
    },
    {
        "query": "Apa kalori dan kalsium di air kelapa muda segar?",
        "expected": "air kelapa muda"
    },
    {
        "query": "Kandungan energi, lemak, dan karbohidrat di biskuit?",
        "expected": "biskuit"
    },
    {
        "query": "Berapa protein dan besi di daging sapi kornet?",
        "expected": "daging sapi kornet"
    },
    {
        "query": "Kandungan gizi lengkap gado-gado per 100 gram?",
        "expected": "gado-gado"
    },
    {
        "query": "Apa serat dan vitamin C di salak medan segar?",
        "expected": "salak medan"
    },
    {
        "query": "Kandungan besi, kalsium, dan vitamin A di genjer segar?",
        "expected": "genjer"
    },
    {
        "query": "Berapa kalori dan protein di bakwan?",
        "expected": "bakwan"
    },
    {
        "query": "Kandungan gizi lengkap rendang sapi masakan per 100 gram?",
        "expected": "rendang sapi"
    },
    {
        "query": "Apa lemak dan natrium di bekasam?",
        "expected": "bekasam"
    },
    {
        "query": "Kandungan energi, serat, dan besi di cabai merah kering?",
        "expected": "cabai merah"
    },
    {
        "query": "Berapa vitamin A dan kalsium di andaliman segar?",
        "expected": "andaliman"
    },
    {
        "query": "Kandungan gizi lengkap kwaci per 100 gram?",
        "expected": "kwaci"
    },
    {
        "query": "Apa protein, besi, dan kalium di wijen mentah?",
        "expected": "wijen"
    },
     {
        "query": "Berapa protein pada paha ayam?",
        "expected": "paha ayam"
    }
]


In [ ]:
import pandas as pd

def soft_match(a, b):
    a_tokens = set(a.split())
    b_tokens = set(b.split())
    return len(a_tokens & b_tokens) / len(a_tokens) >= 0.7


def normalize(text):
    return text.lower().replace(",", "").replace("\n", " ").strip()

def evaluate_retrieval(test_cases, k=3):
    rows = []
    correct = 0

    for i, case in enumerate(test_cases, start=1):
        query = case["query"]

        # ✅ normalize ground truth
        expected = normalize(case["expected"])

        results = hybrid_retrieval(query, k=k)

        # ambil nama hasil retrieval
        retrieved_names = [r["nama"] for r in results]

        # ✅ normalize hasil retrieval
        retrieved_normalized = [normalize(n) for n in retrieved_names]


        # matching (token-based, robust)
        found = any(
            soft_match(expected, name)
            for name in retrieved_normalized
        )

        status = "✅" if found else "❌"

        if found:
            correct += 1

        rows.append({
            "No": i,
            "Query": query,
            "Ground Truth": expected,
            "Top-1": retrieved_names[0] if retrieved_names else "-",
            "Top-K Found": "Ya" if found else "Tidak",
            "Status": status
        })

    accuracy = correct / len(test_cases)
    df = pd.DataFrame(rows)

    return df, accuracy


In [ ]:
df1, acc1 = evaluate_retrieval(test_cases, k=1)
df3, acc3 = evaluate_retrieval(test_cases, k=3)
df5, acc5 = evaluate_retrieval(test_cases, k=5)

print(f"Top-1 Accuracy: {acc1:.2%}")
print(f"Top-3 Accuracy: {acc3:.2%}")
print(f"Top-5 Accuracy: {acc5:.2%}")

display(df3)


Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
Detected intent: nutrition
Detected intent: nutrition
Detected intent: nutrition
Detected intent: food_name
D

,No,Query,Ground Truth,Top-1,Top-K Found,Status
0,1,Berapa kandungan gizi lengkap beras giling men...,beras giling,"Beras Giling, Mentah",Ya,✅
1,2,"Apa saja zat gizi dalam beras hitam mentah, te...",beras hitam,"Beras Hitam, Mentah",Ya,✅
2,3,"Kandungan kalori, protein, dan lemak di beras ...",beras jagung kuning,"Beras Jagung Kuning, Kering, Mentah",Ya,✅
3,4,Berapa besi dan kalsium dalam beras ketan hita...,beras ketan hitam,"Beras Ketan Hitam \nTumbuk, Mentah",Ya,✅
4,5,Kandungan gizi lengkap tempe kedelai murni gor...,tempe kedelai murni,"Tempe Kedelai Murni,\n Mentah",Ya,✅
5,6,Apa protein dan serat di tempe pasar goreng?,tempe pasar,Tempe Pasar Goreng,Ya,✅
6,7,"Kandungan energi, lemak, dan karbohidrat di te...",tempe lamtoro,"Tempe Lamtoro Var,\n Gung Dengan Kulit",Ya,✅
7,8,Berapa vitamin A dan besi dalam ayam goreng Ke...,ayam goreng kentucky dada,"Ayam Goreng\n Kentucky, Dada",Ya,✅
8,9,Kandungan gizi lengkap ayam goreng Church Texa...,ayam goreng church texas dada,"Ayam Goreng Church\n Texas, Dada",Ya,✅
9,10,Apa kalori dan protein di ayam hati segar?,ayam hati,"Ayam, Hati, Segar",Ya,✅


In [ ]:
!pip install --upgrade opentelemetry-api opentelemetry-sdk

In [ ]:
import sys
from sentence_transformers import SentenceTransformer

# --- KONFIGURASI PATH ---
CHROMA_PATH = "/content/drive/MyDrive/skripsian/chroma_db/chroma_tkpi_full"
BASE_PATH = "/content/drive/MyDrive/skripsian/Notebook"

# Tambahkan path agar bisa membaca file ingredient_recommender.py
if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

try:
    from ingredient_recommender import IngredientRecommender
except ImportError as e:
    print(f"Gagal import modul: {e}")

# --- 1. LOAD MODEL EMBEDDING & CHROMADB ---
print("Memuat Model Embedding (intfloat/multilingual-e5-large...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

print("Terhubung ke ChromaDB...\n")
recommender = IngredientRecommender(CHROMA_PATH, embed_model)

Memuat Model Embedding (intfloat/multilingual-e5-large...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Terhubung ke ChromaDB...



In [ ]:
# --- 2. SKENARIO PENGUJIAN ---
tujuan_list = ['cutting', 'bulking', 'maintain']

# --- 2. JALANKAN PERULANGAN ---
for tujuan in tujuan_list:
    print(f"======================================================")
    print(f" 🔍 HASIL PENCARIAN RAG UNTUK TUJUAN : {tujuan.upper()}")
    print(f"======================================================")

    # Pastikan variabel 'recommender' sudah ada (hasil load model sebelumnya)
    basket = recommender.get_balanced_basket(tujuan)

    # Menampilkan hasil Karbohidrat
    print("\n[🍚 KATEGORI: KARBOHIDRAT]")
    if basket['karbo']:
        for i, item in enumerate(basket['karbo'], 1):
            print(f"  {i}. {item['nama']} | Energi: {item.get('kalori', 0)} kkal | Karbo: {item.get('karbo', 0)}g | Serat: {item.get('serat', 0)}g")
    else:
        print("   (Data Kosong)")

    # Menampilkan hasil Protein (Lauk)
    print("\n[🍗 KATEGORI: PROTEIN / LAUK]")
    if basket['protein']:
        for i, item in enumerate(basket['protein'], 1):
            print(f"  {i}. {item['nama']} | Energi: {item.get('kalori', 0)} kkal | Protein: {item.get('protein', 0)}g | Lemak: {item.get('lemak', 0)}g")
    else:
        print("   (Data Kosong)")

    # Menampilkan hasil Serat (Sayur)
    print("\n[🥦 KATEGORI: SAYUR & SERAT]")
    if basket['serat']:
        for i, item in enumerate(basket['serat'], 1):
            print(f"  {i}. {item['nama']} | Energi: {item.get('kalori', 0)} kkal | Serat: {item.get('serat', 0)}g")
    else:
        print("   (Data Kosong)")

    print("\n" + "="*54 + "\n")
for tujuan in tujuan_list:
    print(f"======================================================")
    print(f" 🔍 HASIL PENCARIAN RAG UNTUK TUJUAN : {tujuan.upper()}")
    print(f"======================================================")

    basket = recommender.get_balanced_basket(tujuan)

    # Menampilkan hasil Karbohidrat
    print("\n[🍚 KATEGORI: KARBOHIDRAT]")
    for i, item in enumerate(basket['karbo'], 1):
        # Langsung ambil dari dictionary 'item' yang sudah diproses oleh modul .py
        print(f"  {i}. {item['nama']} | Energi: {item['kalori']} kkal | Karbo: {item['karbo']}g | Serat: {item['serat']}g")

    # Menampilkan hasil Protein (Lauk)
    print("\n[🍗 KATEGORI: PROTEIN / LAUK]")
    for i, item in enumerate(basket['protein'], 1):
        print(f"  {i}. {item['nama']} | Energi: {item['kalori']} kkal | Protein: {item['protein']}g | Lemak: {item['lemak']}g")

    # Menampilkan hasil Serat (Sayur)
    print("\n[🥦 KATEGORI: SAYUR & SERAT]")
    for i, item in enumerate(basket['serat'], 1):
        print(f"  {i}. {item['nama']} | Energi: {item['kalori']} kkal | Serat: {item['serat']}g")

    print("\n" + "="*54 + "\n")

 🔍 HASIL PENCARIAN RAG UNTUK TUJUAN : CUTTING

[🍚 KATEGORI: KARBOHIDRAT]
  1. Beras Merah, Nasi | Energi: 149.0 kkal | Karbo: 32.5g | Serat: 0.3g
  2. Ubi Jalar Merah, Segar | Energi: 151.0 kkal | Karbo: 35.4g | Serat: 0.7g
  3. Batatas Tali, Ubi, Rebus | Energi: 182.0 kkal | Karbo: 42.2g | Serat: 9.2g
  4. Ubi Jalar, Kuning,
 Kukus | Energi: 100.0 kkal | Karbo: 23.8g | Serat: 1.0g
  5. Kaburan, Ubi, Segar | Energi: 133.0 kkal | Karbo: 32.2g | Serat: 0.7g

[🍗 KATEGORI: PROTEIN / LAUK]
  1. Ikan Peda Banjar,
 Mentah | Energi: 156.0 kkal | Protein: 28.0g | Lemak: 4.0g
  2. Ikan Mujahir Pepes | Energi: 121.0 kkal | Protein: 21.7g | Lemak: 2.8g
  3. Ikan Calo/ Peda,
 Mentah | Energi: 81.0 kkal | Protein: 11.4g | Lemak: 1.9g
  4. Ikan Sanggang,
 Masakan | Energi: 240.0 kkal | Protein: 21.7g | Lemak: 7.7g
  5. Ikan Pepetek, Mentah | Energi: 176.0 kkal | Protein: 32.0g | Lemak: 4.4g

[🥦 KATEGORI: SAYUR & SERAT]
  1. Pelecing Kangkung | Energi: 75.0 kkal | Serat: 5.4g
  2. Kangkung, Kukus | En

In [ ]:
import sys
from sentence_transformers import SentenceTransformer

# --- KONFIGURASI PATH ---
CHROMA_PATH = "/content/drive/MyDrive/skripsian/chroma_db/chroma_tkpi_full"
BASE_PATH = "/content/drive/MyDrive/skripsian/Notebook"
if BASE_PATH not in sys.path: sys.path.append(BASE_PATH)

from ingredient_recommender import IngredientRecommender

# --- 1. LOAD MODEL EMBEDDING & CHROMADB ---
print("Memuat Model & Terhubung ke ChromaDB...\n")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
recommender = IngredientRecommender(CHROMA_PATH, embed_model)

Memuat Model & Terhubung ke ChromaDB...



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# --- 2. FUNGSI EVALUATOR GIZI (GROUND TRUTH) ---
def evaluate_relevance(item, category, goal):
    """
    Mengevaluasi apakah makanan yang ditarik ChromaDB sesuai dengan tujuan diet.
    Mengembalikan True (Relevan) atau False (Tidak Relevan).
    """
    kalori = float(item.get('kalori', 0))
    protein = float(item.get('protein', 0))
    lemak = float(item.get('lemak', 0))

    if goal == 'cutting':
        # Cutting: Kalori jangan terlalu tinggi, Lemak harus rendah.
        if category == 'karbo': return kalori <= 250
        if category == 'protein': return lemak <= 15.0 # Jika lemak > 15g (misal gorengan), maka SALAH (False)
        if category == 'serat': return True

    elif goal == 'bulking':
        # Bulking: Butuh padat kalori dan protein tinggi.
        if category == 'karbo': return kalori >= 100
        if category == 'protein': return protein >= 15.0
        if category == 'serat': return True

    elif goal == 'maintain':
        # Maintain: Wajar dan seimbang, lemak jangan ekstrem.
        if category == 'protein': return lemak <= 20.0
        return True

    return True

# --- 3. JALANKAN PENGUJIAN OTOMATIS ---
tujuan_list = ['cutting', 'bulking', 'maintain']
total_all_items = 0
total_all_relevant = 0

print("======================================================")
print(" 🤖 MEMULAI EVALUASI AKURASI RAG (PRECISION SCORE)")
print("======================================================")

for tujuan in tujuan_list:
    basket = recommender.get_balanced_basket(tujuan)
    categories = ['karbo', 'protein', 'serat']

    relevant_count = 0
    total_count = 0

    print(f"\n👉 TARGET: {tujuan.upper()}")

    for cat in categories:
        for item in basket[cat]:
            # Evaluasi item berdasarkan aturan gizi di atas
            is_relevant = evaluate_relevance(item, cat, tujuan)

            # Update statistik
            total_count += 1
            if is_relevant: relevant_count += 1

            # Format tampilan
            status = "✅ RELEVAN" if is_relevant else "❌ TIDAK RELEVAN"
            print(f"   [{cat.upper()}] {item['nama']} (Kal:{item.get('kalori',0)}, Pro:{item.get('protein',0)}, Lem:{item.get('lemak',0)}) -> {status}")

    # Hitung Precision per kategori
    precision = (relevant_count / total_count) * 100
    print(f"   📊 Akurasi {tujuan.upper()}: {relevant_count}/{total_count} Item ({precision:.2f}%)")
    print("-" * 54)

    total_all_items += total_count
    total_all_relevant += relevant_count

# --- 4. HASIL AKHIR KESELURUHAN ---
overall_precision = (total_all_relevant / total_all_items) * 100
print(f"\n======================================================")
print(f" 🎯 TOTAL AKURASI SISTEM RAG (MEAN PRECISION) : {overall_precision:.2f}%")
print(f"======================================================")

 🤖 MEMULAI EVALUASI AKURASI RAG (PRECISION SCORE)

👉 TARGET: CUTTING
   [KARBO] Beras Merah, Nasi (Kal:149.0, Pro:2.8, Lem:0.4) -> ✅ RELEVAN
   [KARBO] Ubi Jalar Merah, Segar (Kal:151.0, Pro:1.6, Lem:0.3) -> ✅ RELEVAN
   [KARBO] Batatas Tali, Ubi, Rebus (Kal:182.0, Pro:2.4, Lem:0.4) -> ✅ RELEVAN
   [KARBO] Ubi Jalar, Kuning,
 Kukus (Kal:100.0, Pro:0.7, Lem:0.3) -> ✅ RELEVAN
   [KARBO] Kaburan, Ubi, Segar (Kal:133.0, Pro:1.0, Lem:0.2) -> ✅ RELEVAN
   [PROTEIN] Ikan Peda Banjar,
 Mentah (Kal:156.0, Pro:28.0, Lem:4.0) -> ✅ RELEVAN
   [PROTEIN] Ikan Mujahir Pepes (Kal:121.0, Pro:21.7, Lem:2.8) -> ✅ RELEVAN
   [PROTEIN] Ikan Calo/ Peda,
 Mentah (Kal:81.0, Pro:11.4, Lem:1.9) -> ✅ RELEVAN
   [PROTEIN] Ikan Sanggang,
 Masakan (Kal:240.0, Pro:21.7, Lem:7.7) -> ✅ RELEVAN
   [PROTEIN] Ikan Pepetek, Mentah (Kal:176.0, Pro:32.0, Lem:4.4) -> ✅ RELEVAN
   [SERAT] Pelecing Kangkung (Kal:75.0, Pro:2.5, Lem:2.8) -> ✅ RELEVAN
   [SERAT] Kangkung, Kukus (Kal:30.0, Pro:3.2, Lem:0.7) -> ✅ RELEVAN
   [SERAT]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Ambil satu contoh item sayur dari hasil target "cutting"
contoh_sayur = recommender.get_balanced_basket('cutting')['serat'][0]

# Tampilkan SEMUA isi data mentahnya
print("Isi Mentah Data Sayur di ChromaDB:")
print(contoh_sayur)

Isi Mentah Data Sayur di ChromaDB:
{'nama': 'Kangkung, segar', 'kalori': 28.0, 'protein': 3.4, 'lemak': 0.7, 'karbo': 3.9, 'desc': 'Nama Bahan: Kangkung, segar. Kategori: MENTAH. Nutrisi per 100g: Energi 28.0 kkal, Protein 3.4g, Lemak 0.7g, Karbohidrat 3.9g.'}
